In [1]:
import numpy as np 
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, SimpleRNN , Dense

I0000 00:00:1787678203.090304   12607 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1787678203.169028   12607 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787678205.495543   12607 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
sentences = [
    "I love this product",
    "This movie made me smile",
    "The service was quick and friendly",
    "Today felt bright and happy",
    "This is the best day",
    "Absolutely fantastic experience",
    "I enjoyed every single moment",
    "Great job, well done",
    "The food tasted delicious",
    "Totally recommend to everyone",
    "Very satisfied with results",
    "This is better than expected",
    "Amazing quality and value",
    "Such a pleasant surprise",
    "I feel positive about this",
    "I am extremely happy with my purchase",
    "The experience was wonderful",
    "Everything worked perfectly",
    "I really appreciate the excellent service",
    "This exceeded all my expectations",
    "The quality is outstanding",
    "I had a fantastic time",
    "The staff was very helpful and polite",
    "I am impressed with the results",
    "This is truly amazing",
    "I would definitely buy this again",
    "The product is exactly what I needed",
    "I am very pleased with the performance",
    "What a wonderful experience",
    "This made my day much better"
]

labels = [1]*15  + [0]*15
labels = np.array(labels)

In [3]:
vocab_size = 2000

In [4]:
tok = Tokenizer(num_words=vocab_size , oov_token= "<00V>")
tok.fit_on_texts(sentences)

In [5]:
seqs = tok.texts_to_sequences(sentences)

In [6]:
max_len = max(len(s) for s in seqs)
X = pad_sequences(seqs , maxlen = max_len , padding = 'post')
y = labels

In [7]:
embed_dim = 16
rnn_unit = 8

In [8]:
inp = Input(
    shape=(max_len,),
    dtype="int32",
    name="input"
)

x = Embedding(
    input_dim=vocab_size,
    output_dim=embed_dim,
    mask_zero=True,
    name="embed"
)(inp)

E0000 00:00:1787678207.778799   12607 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1787678207.779380   16802 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1787678207.799595   12607 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


In [9]:
rnn = SimpleRNN(units=rnn_unit , return_sequences=True , return_state=True , name='simple_rnn')

In [12]:
inp = Input(shape=(max_len,), dtype="int32", name="input")

x = Embedding(
    input_dim=vocab_size,
    output_dim=8,
    mask_zero=True,
    name="embed"
)(inp)

x_output, x_last = rnn(x)

out = Dense(1, activation="sigmoid", name="out")(x_last)

model = Model(inputs=inp, outputs=out)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 7)         │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embed (Embedding)   │ (None, 7, 8)      │     16,000 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal_1         │ (None, 7)         │          0 │ input[0][0]       │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ simple_rnn          │ [(None, 7, 8),    │        200 │ embed[0][0],      │
│ (SimpleRNN)         │ (None, 8)]        │            │ not_equal_1[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ out (Dense)         │ (None, 1)         │          9 │ simple_rnn[2][1]  │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 16,209 (63.32 KB)

 Trainable params: 16,209 (63.32 KB)

 Non-trainable params: 0 (0.00 B)